# Run any chapter's code on Google Colab

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/profrodai/profrodai-resources/blob/main/courses/sovereign-agent-book/book/textbook/appendices/profrod_sovereign_agent_textbook_chapter_code_on_colab_v1.ipynb)

> **Learn with Prof Rod**: *Build Your Always-On AI Agent From Scratch*. Read the full book at [profrod.ai/book](https://profrod.ai/book) and join the [learner community](https://profrod.ai/community).

Every chapter of the book ends with a checkpoint: a script that runs the chapter's code and checks what it claims. This notebook runs any of them on Colab's Python 3.12. It fetches the course, installs the two packages the course pins (the finished agent and Pydantic), and runs the checkpoint you choose. The checkpoints are offline: no model, key or network is needed once the setup cell has run.

## 1. Setup

On Colab this clones the public course repository into `/content` and installs its pinned dependencies. Elsewhere, start this notebook from the course folder with those dependencies installed.

In [ ]:
import os
import subprocess
import sys
import tomllib
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
if ON_COLAB:
    checkout = Path("/content/profrodai-resources")
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/profrodai/profrodai-resources",
             str(checkout)],
            check=True,
        )
    os.chdir(checkout / "courses" / "sovereign-agent-book")
assert Path("book/textbook/checkpoints").is_dir(), "start this notebook from the course folder"
assert sys.version_info[:2] >= (3, 12), "the chapters need Python 3.12 or newer"
pinned = tomllib.loads(Path("pyproject.toml").read_text())["project"]["dependencies"]
try:
    import sovereign_agent  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pinned], check=True)
print("Python", sys.version.split()[0], "in", Path.cwd())
print("Pinned:", *pinned, sep="\n  ")

## 2. Choose a chapter and run its checkpoint

Set `CHAPTER` to any number from 1 to 21. The output is the checkpoint's own report; a failure prints its traceback.

In [ ]:
import json

CHAPTER = 13

book = json.loads(Path("book/textbook/BOOK.json").read_text())
row = next(chapter for chapter in book["chapters"] if chapter["number"] == CHAPTER)
checkpoint = Path("book/textbook") / row["checkpoint"]
print("Chapter", CHAPTER, "-", checkpoint.name)
result = subprocess.run([sys.executable, str(checkpoint)], capture_output=True, text=True)
print(result.stdout[-4000:])
if result.returncode:
    print(result.stderr[-4000:])
    raise SystemExit(f"Chapter {CHAPTER}'s checkpoint failed")

## 3. Run every chapter

This takes a few minutes. It stops at the first failure and names it.

In [ ]:
RUN_ALL = False
if RUN_ALL:
    for row in book["chapters"]:
        checkpoint = Path("book/textbook") / row["checkpoint"]
        result = subprocess.run([sys.executable, str(checkpoint)], capture_output=True, text=True)
        print(f"Chapter {row['number']:2}:", "PASS" if result.returncode == 0 else "FAIL")
        if result.returncode:
            print(result.stderr[-2000:])
            break